In [35]:
import numpy as np
import phate
from scipy.stats import spearmanr
from sklearn import linear_model
from sklearn.model_selection import RepeatedKFold

In [21]:
seed=0
np.random.seed(seed)

print ('Load data...')
trajectory_data = np.load('splatter_simulated_data.npz')
data = trajectory_data['data']
NCELLS = data.shape[0]

print ('Uniform signal...')
uniform_signal = np.ones((1, NCELLS))
uniform_signal = uniform_signal / np.linalg.norm(uniform_signal, axis=1).reshape(-1,1)

print ('Build cellular graph...')
phate_op = phate.PHATE(random_state=seed, use_pygsp=True, n_jobs=-1)
phate_op.fit(data)
G = phate_op.graph

Load data...
Uniform signal...
Build cellular graph...
Running PHATE on 10000 observations and 8821 variables.
Calculating graph and diffusion operator...
  Calculating PCA...
  Calculated PCA in 5.45 seconds.
  Calculating KNN search...
  Calculated KNN search in 2.99 seconds.
  Calculating affinities...
  Calculated affinities in 2.73 seconds.
Calculated graph and diffusion operator in 11.57 seconds.
Calculating landmark operator...
  Calculating SVD...
  Calculated SVD in 2.28 seconds.
  Calculating KMeans...
  Calculated KMeans in 6.52 seconds.
Calculated landmark operator in 10.14 seconds.


In [22]:
## TASK SETUP, learn coexpression embedding and localization embedding
task = 'coexpression'
if task == 'coexpression':
    signals = data.T
    signals = signals / np.linalg.norm(signals, axis=1).reshape(-1,1)

elif task == 'localization':
    signals = np.load('localization_signals.npz')['signals']
    signals = signals / np.linalg.norm(signals, axis=1).reshape(-1,1)

In [24]:
## INPUT YOUR EMBEDDINGS
coexpression_embedding = np.load('results/coexpression/GSPA_QR/0_results.npz')['signal_embedding']
localization_embedding = np.load('results/localization/GSPA_QR/0_results.npz')['signal_embedding']

In [27]:
## CONFIRM DIMENSIONS MATCH EXPECTED

print(coexpression_embedding.shape) ## should be (8821, output_dim)
print(localization_embedding.shape) ## should be (250, output_dim)

(8821, 128)
(250, 128)


### Coexpression embedding evaluation

In [37]:
print ('Stratify Spearman correlation...')
true_counts = trajectory_data['true_counts']
true_lib_size = true_counts.T.sum(axis=1)
spearman_res = spearmanr(true_counts)
np.fill_diagonal(spearman_res.correlation, 0)
corr_bins = np.linspace(spearman_res.correlation.min(), spearman_res.correlation.max(), 4)

min_bin_size = float('inf')
for i,corr in enumerate(corr_bins):
    if i == 0: continue
    choices = np.array(np.where((spearman_res.correlation > corr_bins[i-1]) & (spearman_res.correlation < corr) & (spearman_res.correlation != 0))).T
    if choices.shape[0] < min_bin_size:
        min_bin_size = choices.shape[0]

spearmans = []

print ('Stratify library size...')
min_lib_size= float('inf')
for i,corr in enumerate(corr_bins):

    choices_bin = []
    if i == 0: continue

    ## res.correlation does not equal zero, excluding self edges
    choices = np.array(np.where((spearman_res.correlation > corr_bins[i-1]) & (spearman_res.correlation < corr) & (spearman_res.correlation != 0))).T
    
    lib_size_mean_per_pair = np.vstack((true_lib_size[choices[:, 0]], true_lib_size[choices[:, 1]])).mean(axis=0)
    lib_size = np.linspace(lib_size_mean_per_pair.min(), lib_size_mean_per_pair.max(), 3)

    for j,bin in enumerate(lib_size):
        if j == 0: continue
        choices_ = np.array(np.where((lib_size_mean_per_pair > lib_size[j-1]) & (lib_size_mean_per_pair < bin))).T
        if choices_.shape[0] < min_lib_size:
            min_lib_size = choices_.shape[0]

print ('Sample by stratification...')
samples = []
for i,corr in enumerate(corr_bins):

    choices_bin = []
    if i == 0: continue

    ## res.correlation does not equal zero, excluding self edges
    choices = np.array(np.where((spearman_res.correlation > corr_bins[i-1]) & (spearman_res.correlation < corr) & (spearman_res.correlation != 0))).T
    
    lib_size_mean_per_pair = np.vstack((true_lib_size[choices[:, 0]], true_lib_size[choices[:, 1]])).mean(axis=0)
    lib_size = np.linspace(lib_size_mean_per_pair.min(), lib_size_mean_per_pair.max(), 3)

    for j,bin in enumerate(lib_size):
        if j == 0: continue
        choices_ = np.array(np.where((lib_size_mean_per_pair > lib_size[j-1]) & (lib_size_mean_per_pair < bin))).T
        choices_bin.append(choices_[np.random.choice(choices_.shape[0], size=min_lib_size, replace=False, )])

    samples.append(choices[np.vstack(choices_bin).flatten()])

samples = np.vstack(samples)
kf = RepeatedKFold(n_splits=2, n_repeats=10)
splits = kf.split(samples)

for (train_index, test_index) in splits:
    X_train = []
    y_train = []
    X_test = []
    y_test = []

    train_index = samples[train_index]
    test_index = samples[test_index]

    for (a,b) in train_index:
        X_train.append(np.hstack((coexpression_embedding[a], coexpression_embedding[b])))
        y_train.append(spearman_res.correlation[a][b])
    for (a,b) in test_index:
        X_test.append(np.hstack((coexpression_embedding[a], coexpression_embedding[b])))
        y_test.append(spearman_res.correlation[a][b])

    X_train = np.array(X_train)
    X_test = np.array(X_test)
    y_train = np.array(y_train)
    y_test = np.array(y_test)

    regr = linear_model.Ridge()
    regr.fit(X_train, y_train)
    y_pred = regr.predict(X_test)

    spearmans.append(spearmanr(y_test, y_pred).correlation)

print ('Spearman', np.median(spearmans))

Stratify Spearman correlation...
Stratify library size...
Sample by stratification...
Spearman 0.5903401008928162


### Localization evaluation

In [41]:
labels_y = np.load(f'./data/localization_signals.npz')['spread']
spearmans = []
kf = RepeatedKFold(n_splits=2, n_repeats=10)
splits = kf.split(localization_embedding)

for (train_index, test_index) in splits:
    X_train = localization_embedding[train_index]
    X_test = localization_embedding[test_index]
    y_train = labels_y[train_index]
    y_test = labels_y[test_index]

    regr = linear_model.Ridge()
    regr.fit(X_train, y_train)
    y_pred = regr.predict(X_test)

    spearmans.append(spearmanr(y_test, y_pred).correlation)

print('Spearman',np.median(spearmans))

Spearman 0.9312000966590629
